# Bronze Layer — Ingest TPC-H Source Tables

Reads the eight TPC-H source tables from `samples.tpch` and writes one Bronze Delta table per source into `dev.bronze`, preserving all source columns and grain. Each table carries ingestion metadata (source file path, source file name, ingestion timestamp, pipeline run ID). The pipeline is idempotent — every run fully overwrites the Bronze targets, so running it twice produces the same result as running it once — and verifies Bronze row counts against the source.

In [0]:
# ── Configuration ──────────────────────────────────────────────────────────────
dbutils.widgets.text("CATALOG", "dev")
CATALOG = dbutils.widgets.get("CATALOG")

SOURCE_CATALOG = "samples"
SOURCE_SCHEMA = "tpch"
TARGET_SCHEMA = f"{CATALOG}.bronze"

# One run ID per pipeline execution — stamped on every row for traceability.
from datetime import datetime, timezone

RUN_ID = f"tpch_ingest_{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}"
INGESTED_AT = datetime.now(timezone.utc)

# (table_name, business meaning)
SOURCES = [
    ("customer", "Customer master and market segment"),
    ("orders",   "Customer orders, dates, and status"),
    ("lineitem", "Order lines with quantities, prices, discounts, taxes, and shipping dates"),
    ("part",     "Product/part master"),
    ("partsupp", "Supplier-to-product supply relationships and costs"),
    ("supplier", "Supplier master"),
    ("nation",   "Nation and region mapping"),
    ("region",   "Region master"),
]

print(f"Catalog           : {CATALOG}")
print(f"Source            : {SOURCE_CATALOG}.{SOURCE_SCHEMA}")
print(f"Target schema     : {TARGET_SCHEMA}")
print(f"Pipeline run ID   : {RUN_ID}")
print(f"Ingested at (UTC) : {INGESTED_AT}")
print(f"Tables to ingest  : {len(SOURCES)}")

In [0]:
# ── Ingestion: samples.tpch → Bronze Delta ─────────────────────────────────────
# Idempotent: every run fully overwrites the target table, so running twice
# yields the same rows as running once (only the metadata columns are refreshed).
from pyspark.sql import functions as F

row_count_results = []

for table, comment in SOURCES:
    source_ref = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{table}"
    target = f"{TARGET_SCHEMA}.{table}"

    # 1. Read the source table (read-only — never written back).
    df = spark.table(source_ref)
    source_count = df.count()

    # 2. Preserve all source columns and grain; append ingestion metadata.
    #    The sources are Unity Catalog tables, not raw files, so the metadata
    #    columns carry the source table reference in path/filename form.
    bronze_df = (
        df
        .withColumn("_ingestion_timestamp", F.lit(INGESTED_AT).cast("timestamp"))
        .withColumn("_source_file_path", F.lit(f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}"))
        .withColumn("_source_file_name", F.lit(f"{table}"))
        .withColumn("_pipeline_run_id", F.lit(RUN_ID))
    )

    # 3. Overwrite the Bronze table (full replace — rerunnable by design).
    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target)
    )

    spark.sql(f"COMMENT ON TABLE {target} IS '{comment}'")

    # 4. Verify: Bronze row count must match the source row count.
    bronze_count = spark.table(target).count()
    row_count_results.append((table, source_ref, source_count, bronze_count))

    status = "✓" if source_count == bronze_count else "✗ MISMATCH"
    print(f"{status} {table:<10} source={source_count:>10,}  bronze={bronze_count:>10,}")

print(f"\nIngestion complete — run ID: {RUN_ID}")

In [0]:
# ── Verification: Bronze vs. source row counts ────────────────────────────────
verification = spark.createDataFrame(
    row_count_results,
    ["bronze_table", "source_table", "source_rows", "bronze_rows"],
)
verification = verification.withColumn(
    "match", F.col("source_rows") == F.col("bronze_rows")
)
display(verification.orderBy("bronze_table"))

all_match = verification.filter(F.col("match") == False).count() == 0
print(f"\n{'ALL 8 TABLES MATCH SOURCE ROW COUNTS' if all_match else 'ROW COUNT MISMATCHES FOUND — investigate above'}")

# Sanity check: one Bronze table keeps every source column plus the 4 metadata columns.
sample_cols = set(spark.table(f"{TARGET_SCHEMA}.customer").columns)
expected_meta = {"_ingestion_timestamp", "_source_file_path", "_source_file_name", "_pipeline_run_id"}
sample_source_cols = set(spark.table(f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.customer").columns)
print(f"Source columns preserved : {sample_source_cols.issubset(sample_cols)}")
print(f"Metadata columns present : {expected_meta.issubset(sample_cols)}")